## AUTOMATIZACION DE ANEXOS - DERIVAS - FACTOR DE PARTICIPACION - MASA DE PISO - REACCIONES BASALES - PERIODOS

In [102]:
%pip install comtypes
%pip install xlwings

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Librerias 

In [103]:
import comtypes.client
import xlwings as xw
import pandas as pd
import numpy as np

Conectamos el modelo de etabs en el que trabajaremos

In [104]:
# conectarse a la instancia activa de etabs
helper = comtypes.client.CreateObject('ETABSv1.Helper')  # Creamos un objeto para conectarnos a etabs, 'ETABSv1.Helper'es el ayudando de etabs para esta tarea
programa_abierto= helper.GetObject("CSI.ETABS.API.ETABSObject") # busca el programa de etabs que este abierto

if programa_abierto is None:
    raise RuntimeError("No se encontró ninguna instancia de ETABS abierta. Verifica que ETABS esté corriendo con un modelo cargado.")
etabs_model= programa_abierto.SapModel   # extrae la propiedad mas importante del objeto de etabs.
etabs_model.SetPresentUnits(6)
print("conectado a:", etabs_model.GetModelFilename())
print(programa_abierto)

conectado a: C:\Users\andre\Documents\ATRES_INGENIERIA\TORRRE APARTAMENTE HACIENDA CANNAN\02-MODELO\2026-07-27 MODELO HACIENDA CANAAN_alejo.EDB
<POINTER(cOAPI) ptr=0x19d0bd1fda0 at 19d27dd6dd0>


# MASA DE PISO


Conectamos el excel donde imprimirelos la informacion de masa de piso

In [105]:
# Conectarte por nombre exacto del archivo (con o sin extensión, ambos funcionan)
excel_libro = xw.Book('05-MASA DE LA ESTRUCTURA HACIENDA CANNAN.xlsx') # abre y conecta el libro de excel
# Seleccionar la hoja donde vas a trabajar
hoja_masa = excel_libro.sheets['Mass Summary by Story' ]  

In [106]:
#Masa_tabla = "Mass Summary by Story" # Nombre de la tabla donde se extrae la info

#Extraemos la tabla completa
version_tabla_masa = 0 # version de la tabla 
nombres_colums_masa = [] # guardamos el nombre de las columnas
num_filas_masa = 0 # guardamos el numero de filas
TableData_masa = [] # informacion completa de la tabla masa de etabs
 
#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, _, nombres_colums_masa, _, TableData_masa, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( "Mass Summary by Story", [],"", 0, [], 0, TableData_masa   )
#print(nombres_colums_masa)   # nombres de columnas            #################### PODRIA SOLO IMPORTAR OUTCASE Y UX ####################
#print(num_filas)            # cuántas filas trajo


n_cols = len(nombres_colums_masa) # cantidad de columnas extraidas de la tabla
array_masa = np.array(TableData_masa).reshape(-1, n_cols)  # convertimos la TableData_masa en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
masa_tabla = pd.DataFrame(array_masa, columns=nombres_colums_masa) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

masa_tabla.head() # muestra una parte de la tabla.

# Extramos la masa por piso

masa_final = masa_tabla[['Story', 'UX','UY','UZ']].copy()

masa_final
#hoja_masa.range('A4').options(pd.DataFrame, index=False, header=False).value = masa_final
hoja_masa.range('A4').value = masa_final.values

# FACTOR DE PARTICIPACION DE MASA

In [107]:
# Conectarte por nombre exacto del archivo (con o sin extensión, ambos funcionan)
excel_modal = xw.Book('04-FACTOR DE PARTICIPACIÓN MODAL HACIENDA CANNAN.xlsx') # abre y conecta el libro de excel
# Seleccionar la hoja donde vas a trabajar
hoja_participacion= excel_modal.sheets['Modal Participating Mass Ratios' ]  

In [108]:
#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, _, nombres_colums_modal, _, TableData_modal, ret = etabs_model.DatabaseTables.GetTableForDisplayArray("Modal Participating Mass Ratios", [],"", 0, [], 0, [] )

n_cols = len(nombres_colums_modal) # cantidad de columnas extraidas de la tabla
array_modal = np.array(TableData_modal).reshape(-1, n_cols)  # convertimos la TableData_modal en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
modal_tabla = pd.DataFrame(array_modal, columns=nombres_colums_modal) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

modal_tabla.head() # muestra una parte de la tabla.

# Extramos la masa por piso

#insertamos el pandas en la hoja de excel.
hoja_participacion.range('A4').options(pd.DataFrame, index=False, header=False).value = modal_tabla # pego la tabla de modal


# DERIVAS DE PISO

In [109]:
# Conectarte por nombre exacto del archivo (con o sin extensión, ambos funcionan)
derivas_libro = xw.Book('03-DERIVAS DE PISO HACIENDA CANNAN.xlsx') # abre y conecta el libro de excel
# Seleccionar la hoja donde vas a trabajar
hoja_story_drift= derivas_libro.sheets['Story Drifts']  
hoja_joint_drift= derivas_libro.sheets['Joint Drifts'] 

In [110]:

etabs_model.DatabaseTables.SetLoadCombinationsSelectedForDisplay(["U_B.2.4-5_1 DER_H","U_B.2.4-5_1 DER","U_B.2.4-5_2 DER_H","U_B.2.4-5_2 DER","U_B.2.4-5_3 DER_H",
    "U_B.2.4-5_3 DER","U_B.2.4-5_4 DER_H","U_B.2.4-5_4 DER","U_B.2.4-7_1 DER_H","U_B.2.4-7_1 DER","U_B.2.4-7_2 DER_H","U_B.2.4-7_2 DER","U_B.2.4-7_3 DER_H","U_B.2.4-7_3 DER",
"U_B.2.4-7_4 DER_H","U_B.2.4-7_4 DER"])

etabs_model.DatabaseTables.SetLoadCasesSelectedForDisplay([])

#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, _, nombres_colums_drifts, _, TableData_drifts, ret = etabs_model.DatabaseTables.GetTableForDisplayArray("Story Drifts", [],"", 0, [], 0, []  )

n_cols = len(nombres_colums_drifts) # cantidad de columnas extraidas de la tabla
array_drifts = np.array(TableData_drifts).reshape(-1, n_cols)  # convertimos la TableData_modal en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
drifts_tabla = pd.DataFrame(array_drifts, columns=nombres_colums_drifts) # convertimos el array a un pandas, tabla excel con sus nombres de columnas


## Extramos la masa por piso
df_drifts = drifts_tabla[['Story', 'OutputCase','CaseType','Direction','Drift','Label','X','Y','Z']].copy() #extraigo lo que necesito 

hoja_story_drift.range('A4').options(pd.DataFrame,index = False, header = False).value = df_drifts
#print(df_drifts)
#drifts_tabla.head() # muestra una parte de la tabla.

In [111]:
etabs_model.DatabaseTables.SetLoadCombinationsSelectedForDisplay(["U_B.2.4-5_1 DER_H","U_B.2.4-5_1 DER","U_B.2.4-5_2 DER_H","U_B.2.4-5_2 DER","U_B.2.4-5_3 DER_H",
    "U_B.2.4-5_3 DER","U_B.2.4-5_4 DER_H","U_B.2.4-5_4 DER","U_B.2.4-7_1 DER_H","U_B.2.4-7_1 DER","U_B.2.4-7_2 DER_H","U_B.2.4-7_2 DER","U_B.2.4-7_3 DER_H","U_B.2.4-7_3 DER",
"U_B.2.4-7_4 DER_H","U_B.2.4-7_4 DER"])

etabs_model.DatabaseTables.SetLoadCasesSelectedForDisplay([])

#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, _, nombres_colums_joint, _, TableData_joint, ret = etabs_model.DatabaseTables.GetTableForDisplayArray("Joint Drifts", [],"", 0, [], 0, []  )

n_cols = len(nombres_colums_joint) # cantidad de columnas extraidas de la tabla
array_joint= np.array(TableData_joint).reshape(-1, n_cols)  # convertimos la TableData_modal en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
joint_tabla = pd.DataFrame(array_joint, columns=nombres_colums_joint) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

## Extramos la masa por piso
df_joint = joint_tabla[['Story', 'Label','UniqueName','OutputCase','DispX','DispY','DriftX','DriftY']].copy() #extraigo lo que necesito 

hoja_joint_drift.range('A4').options(pd.DataFrame,index = False, header = False).value = df_joint

#joint_tabla.head() # muestra una parte de la tabla.
#print(df_joint)



# BASE REACTIONS

In [112]:
# Conectarte por nombre exacto del archivo (con o sin extensión, ambos funcionan)
basales_libro = xw.Book('07-REACCIONES BASALES HACIENDA CANNAN.xlsx') # abre y conecta el libro de excel
# Seleccionar la hoja donde vas a trabajar
hoja_basales= basales_libro.sheets['Base Reactions']  

In [113]:
etabs_model.DatabaseTables.SetLoadCasesSelectedForDisplay(['FHEX','FHEY','SPECX','SPECY','ELASX','ELASY']) # Casos de carga que queremos extraer
etabs_model.DatabaseTables.SetLoadCombinationsSelectedForDisplay([]) # NO QUEREMOS NINGUNA COMBINACION
#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, _, nombres_colums_BR, _, TableData_BR, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( "Base Reactions", [],"", 0, [],0, [] )


n_cols = len(nombres_colums_BR) # cantidad de columnas extraidas de la tabla
array_BR = np.array(TableData_BR).reshape(-1, n_cols)  # convertimos la TableData_BR en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
BR_tabla = pd.DataFrame(array_BR, columns=nombres_colums_BR) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

BR_tabla.head() # muestra una parte de la tabla.
df_BR = BR_tabla.copy() # copiamos la tabla

#Escribimos los datos de las FHE en la hoja de excel.
hoja_basales.range('A4').options(pd.DataFrame, index=False, header=False).value = df_BR



# PERIODOS MODALES

In [114]:
# Conectarte por nombre exacto del archivo (con o sin extensión, ambos funcionan)
Periodos_libro = xw.Book('06-PERIODOS HACIENDA CANNAN.xlsx') # abre y conecta el libro de excel
# Seleccionar la hoja donde vas a trabajar
hoja_Periodos= Periodos_libro.sheets['Hoja1']  

In [115]:
#extraemos toda la informacion de la tabla ( tener cuidado con el orden de las variables que extraen )
_, _, nombres_colums_Periodos, _, TableData_Periodos, ret = etabs_model.DatabaseTables.GetTableForDisplayArray( "Modal Periods And Frequencies", [],"", 0, [],0, [] )


n_cols = len(nombres_colums_Periodos) # cantidad de columnas extraidas de la tabla
array_Periodos= np.array(TableData_Periodos).reshape(-1, n_cols)  # convertimos la TableData_BR en un array 2D. Y se acomodan los datos en ncolumnas (n_cols), el (-1) dice que se se formen las filas necesarias. 
Periodos_tabla = pd.DataFrame(array_Periodos, columns=nombres_colums_Periodos) # convertimos el array a un pandas, tabla excel con sus nombres de columnas

Periodos_tabla.head() # muestra una parte de la tabla.
df_Periodos = Periodos_tabla.copy() # copiamos la tabla

#Escribimos los datos de las FHE en la hoja de excel.
hoja_Periodos.range('A4').options(pd.DataFrame, index=False, header=False).value = df_Periodos